# Gravando o resultado: a política de escrita

## Demanda da aula 5

O time de plataforma da Arena Analytics chegou com o último pedido do projeto: **colocar o pipeline em produção**.

Até aqui, tudo o que calculamos viveu dentro de notebooks e sumiu quando a sessão terminou. Em produção, o pipeline
precisa rodar sozinho, chamado por um agendador, e **gravar** os resultados num lugar em que outros sistemas (o
painel da transmissão, o time de analytics) consigam ler. E precisa gravar de forma **segura**: sem perder dado e sem
duplicar dado, mesmo se rodar duas vezes.

E o primeiro teste já tem data: o fornecedor mandou os dados da **Copa de 2026**, que precisam entrar no histórico.

| Etapa | O que fazer | Notebook |
|---|---|---|
| Task 1 | Definir como o pipeline grava: os modos de escrita e os acidentes a evitar | `01_modos_de_escrita` |
| Task 2 | Escolher o formato em que os resultados ficam guardados | `02_formatos` |
| Task 3 | Organizar a saída por Copa e carregar uma Copa nova sem apagar o histórico | `03_partitionby_e_pruning` |
| Task 4 | Colocar o pipeline em produção, rodando fora do notebook, e carregar a Copa de 2026 | `04_locais_e_pipeline_final` |

**Pronto quando:** o pipeline roda com um comando, fora do notebook; grava em Delta Lake, organizado por Copa; carrega
a Copa de 2026 sem apagar nem duplicar o histórico, mesmo se rodar duas vezes; e a saída é conferida automaticamente.

## Task 1: definir como o pipeline grava

In [ ]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-escrita")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
OUT = "../../data/out"

spark

## As partidas do projeto

`ler_partidas(pasta)`: a mesma leitura serve para o histórico e para a Copa de 2026, que chega em outra pasta.

A próxima célula já vem pronta: ela só apaga as pastas que este notebook grava, para ele poder rodar de novo.

In [ ]:
import shutil
for pasta in ["partidas", "teste_acidente"]:
    shutil.rmtree(f"{OUT}/{pasta}", ignore_errors=True)

## E se o job rodar de novo?

| Modo | Se o destino já existe |
|---|---|
| `append` | acrescenta arquivos novos |
| `overwrite` | apaga e grava de novo |

Sem modo nenhum, o padrão é dar **erro** se a pasta já existe: o projeto sempre declara o modo.

**Primeiro acidente:** compare a contagem com a do histórico. Um job com `append` que roda duas vezes grava o dado
duas vezes, sem erro.

## O segundo acidente: sobrescrever a pasta de onde você está lendo

Confira a pasta depois da falha. Por causa da lazy evaluation, a leitura só acontece quando a escrita começa,
e a escrita começa apagando a pasta. A documentação avisa: no `overwrite`, o dado é apagado antes de o novo ser
gravado, e os modos de escrita não são atômicos.

## A política de escrita do projeto

1. **Todo `write` diz o modo.**
2. **Nada de `append` num job que pode rodar de novo.**
3. **Nunca `overwrite` no caminho de onde se está lendo.**

O Parquet não desfaz uma escrita que falhou: o formato do vídeo 5.2 resolve. E a Copa de 2026? O `append` duplica e
o `overwrite` apaga o histórico: a resposta vem no vídeo 5.3.

## Resultado da task 1

Política de escrita definida. Próxima task: o formato dos arquivos.

## Para consultar depois

- **Os outros dois modos:** `error` (ou `errorifexists`), o padrão, lança erro se o destino existe; `ignore` não faz
  nada se o destino existe, sem avisar.
- **A mesma escrita, de outro jeito:** `df.write.format("parquet").mode("overwrite").save(caminho)` é o mesmo que
  `df.write.mode("overwrite").parquet(caminho)`. É esse jeito, com `format(...)`, que o vídeo 5.2 usa.
- **Arquivos ocultos:** junto com os `part-`, o Spark grava arquivos `.crc`, de verificação, que o `ls` não mostra.
- **Ler uma pasta** lê todos os arquivos de dados dentro dela; arquivos que começam com `_` ou `.` são ignorados.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()